In [2]:
import pandas as pd
import numpy as np
import re  
from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import SimpleRNN, Dense, Embedding

In [3]:
data = pd.read_csv('swiggy.csv')
print("Columns in the dataset:")
print(data.columns.tolist())

Columns in the dataset:
['ID', 'Area', 'City', 'Restaurant Price', 'Avg Rating', 'Total Rating', 'Food Item', 'Food Type', 'Delivery Time', 'Review']


In [4]:
data["Review"] = data["Review"].str.lower()
data["Review"] = data["Review"].replace(r'[^a-z0-9\s]', '', regex=True)

data['sentiment'] = data['Avg Rating'].apply(lambda x: 1 if x > 3.5 else 0)
data = data.dropna()

In [5]:
max_features = 5000
max_length = 200

tokenizer = Tokenizer(num_words=max_features)
tokenizer.fit_on_texts(data["Review"])
X = pad_sequences(tokenizer.texts_to_sequences(
    data["Review"]), maxlen=max_length)
y = data['sentiment'].values

In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
X_train, X_val, y_train, y_val = train_test_split(
    X_train, y_train, test_size=0.1, random_state=42, stratify=y_train
)

In [7]:
model = Sequential([
    Embedding(input_dim=max_features, output_dim=16, input_length=max_length),
    SimpleRNN(64, activation='tanh', return_sequences=False),
    Dense(1, activation='sigmoid')
])

model.compile(
    loss='binary_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

c:\Users\HAK3\AppData\Local\Programs\Python\Python313\Lib\site-packages\keras\src\layers\core\embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [8]:
history = model.fit(
    X_train, y_train,
    epochs=5,
    batch_size=32,
    validation_data=(X_val, y_val),
    verbose=1
)

score = model.evaluate(X_test, y_test, verbose=0)
print(f"Test accuracy: {score[1]:.2f}")

Epoch 1/5
180/180 ━━━━━━━━━━━━━━━━━━━━ 13s 47ms/step - accuracy: 0.7059 - loss: 0.6018 - val_accuracy: 0.7156 - val_loss: 0.5984
Epoch 2/5
180/180 ━━━━━━━━━━━━━━━━━━━━ 11s 51ms/step - accuracy: 0.7160 - loss: 0.5971 - val_accuracy: 0.7156 - val_loss: 0.6010
Epoch 3/5
180/180 ━━━━━━━━━━━━━━━━━━━━ 7s 41ms/step - accuracy: 0.7135 - loss: 0.5988 - val_accuracy: 0.7156 - val_loss: 0.6001
Epoch 4/5
180/180 ━━━━━━━━━━━━━━━━━━━━ 6s 35ms/step - accuracy: 0.7160 - loss: 0.5977 - val_accuracy: 0.7156 - val_loss: 0.5970
Epoch 5/5
180/180 ━━━━━━━━━━━━━━━━━━━━ 7s 37ms/step - accuracy: 0.7160 - loss: 0.5963 - val_accuracy: 0.7156 - val_loss: 0.6098
Test accuracy: 0.72


In [9]:
history = model.fit(
    X_train, y_train,
    epochs=5,
    batch_size=32,
    validation_data=(X_val, y_val),
    verbose=1
)

score = model.evaluate(X_test, y_test, verbose=0)
print(f"Test accuracy: {score[1]:.2f}")

Epoch 1/5
180/180 ━━━━━━━━━━━━━━━━━━━━ 9s 49ms/step - accuracy: 0.7160 - loss: 0.5965 - val_accuracy: 0.7156 - val_loss: 0.5989
Epoch 2/5
180/180 ━━━━━━━━━━━━━━━━━━━━ 8s 46ms/step - accuracy: 0.7160 - loss: 0.5973 - val_accuracy: 0.7156 - val_loss: 0.5960
Epoch 3/5
180/180 ━━━━━━━━━━━━━━━━━━━━ 10s 45ms/step - accuracy: 0.7160 - loss: 0.5975 - val_accuracy: 0.7156 - val_loss: 0.5961
Epoch 4/5
180/180 ━━━━━━━━━━━━━━━━━━━━ 10s 58ms/step - accuracy: 0.7160 - loss: 0.5963 - val_accuracy: 0.7156 - val_loss: 0.5995
Epoch 5/5
180/180 ━━━━━━━━━━━━━━━━━━━━ 9s 53ms/step - accuracy: 0.7160 - loss: 0.5968 - val_accuracy: 0.7156 - val_loss: 0.5960
Test accuracy: 0.72


In [10]:
from transformers import pipeline

sentiment_pipeline = pipeline('sentiment-analysis', model='nlptown/bert-base-multilingual-uncased-sentiment')

text = "The movie was absolutely fantastic! I loved every moment of it."

result = sentiment_pipeline(text)[0]

label_map = {
    '1 star': 'Very Negative',
    '2 stars': 'Negative',
    '3 stars': 'Neutral',
    '4 stars': 'Positive',
    '5 stars': 'Very Positive'
}

custom_label = label_map[result['label']]
confidence_percentage = result['score'] * 100

print(f"Sentiment: {custom_label}")
print(f"Confidence: {confidence_percentage:.2f}%")

c:\Users\HAK3\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 201/201 [00:00<00:00, 1166.45it/s]


Sentiment: Very Positive
Confidence: 95.22%


In [11]:
from transformers import pipeline

emotion_pipeline = pipeline('text-classification', model='j-hartmann/emotion-english-distilroberta-base')

text = "I am so excited about the new project!"

result = emotion_pipeline(text)[0]

custom_label = result['label'].capitalize()
confidence_percentage = result['score'] * 100

print(f"Emotion: {custom_label}")
print(f"Confidence: {confidence_percentage:.2f}%")

Loading weights: 100%|██████████| 105/105 [00:00<00:00, 1881.54it/s]
RobertaForSequenceClassification LOAD REPORT from: j-hartmann/emotion-english-distilroberta-base
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Emotion: Joy
Confidence: 97.58%


In [13]:
import sys
!{sys.executable} -m pip install spacy nltk
!{sys.executable} -m spacy download en_core_web_sm
import nltk
nltk.download('vader_lexicon')

   ---------------------------------------- 0.0/14.2 MB ? eta -:--:--
   ---------------------------------------- 0.0/14.2 MB ? eta -:--:--
   ---------------------------------------- 0.0/14.2 MB ? eta -:--:--
   ---------------------------------------- 0.0/14.2 MB ? eta -:--:--
   ---------------------------------------- 0.0/14.2 MB ? eta -:--:--
   ---------------------------------------- 0.0/14.2 MB ? eta -:--:--
    --------------------------------------- 0.3/14.2 MB ? eta -:--:--
    --------------------------------------- 0.3/14.2 MB ? eta -:--:--
    --------------------------------------- 0.3/14.2 MB ? eta -:--:--
    --------------------------------------- 0.3/14.2 MB ? eta -:--:--
    --------------------------------------- 0.3/14.2 MB ? eta -:--:--
    --------------------------------------- 0.3/14.2 MB ? eta -:--:--
    --------------------------------------- 0.3/14.2 MB ? eta -:--:--
    --------------------------------------- 0.3/14.2 MB ? eta -:--:--
    ----------------

[nltk_data] Downloading package vader_lexicon to
[nltk_data]     C:\Users\HAK3\AppData\Roaming\nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!


True

In [14]:
import spacy
from spacy.matcher import Matcher
from nltk.sentiment import SentimentIntensityAnalyzer 
import nltk
nltk.download('vader_lexicon')

nlp = spacy.load("en_core_web_sm")

text = "The food was delicious, but the service was slow."

doc = nlp(text)

matcher = Matcher(nlp.vocab)
pattern = [{"LOWER": {"IN": ["food", "service"]}}] 
matcher.add("AspectMatcher", [pattern])


sia = SentimentIntensityAnalyzer()  
aspects = []
for match_id, start, end in matcher(doc):
    aspect = doc[start:end].text
    sentiment = None
    
    scores = sia.polarity_scores(aspect)
    sentiment = "positive" if scores["compound"] > 0 else "negative" 

    aspects.append({"aspect": aspect, "sentiment": sentiment})

print(aspects)

[nltk_data] Downloading package vader_lexicon to
[nltk_data]     C:\Users\HAK3\AppData\Roaming\nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!


[{'aspect': 'food', 'sentiment': 'negative'}, {'aspect': 'service', 'sentiment': 'negative'}]


In [15]:
from transformers import pipeline

multilingual_sentiment_pipeline = pipeline('sentiment-analysis', model='nlptown/bert-base-multilingual-uncased-sentiment')

texts = [
    "The service was excellent.",
    "The movie was very disappointing."
]

results = [multilingual_sentiment_pipeline(text) for text in texts]
for text, result in zip(texts, results):
    print(f"Text: {text}\nSentiment: {result}\n")

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 1319.72it/s]


Text: The service was excellent.
Sentiment: [{'label': '5 stars', 'score': 0.7189083695411682}]

Text: The movie was very disappointing.
Sentiment: [{'label': '2 stars', 'score': 0.49748459458351135}]

